#### 1. 统计各个数据集每个查询的核心实例个数(就是/home/wangshuo/resource/datasets/parler_data/{dataset_name}/results/aggregated_results文件夹下各文件的元组数量(满足estimateW > 0))

只统计出现在 T_true_*.json 文件中的查询

In [4]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json

def count_core_instances(dataset_name: str,
                         base_dir: str = "/home/wangshuo/resource/datasets/parler_data"):
    """
    统计每个查询的核心实例个数：
    核心实例 = aggregated_results 中 estimateW > 0 (或 a > 0) 的行数
    【过滤条件】：只统计在 ground_truth JSON 文件中存在的查询。
    """
    # 1. 路径定义
    agg_dir = os.path.join(base_dir, dataset_name, "results", "aggregated_results")
    gt_path = os.path.join(base_dir, dataset_name, "results", "T_true_ML1_oracle2_probability_ML2_oracle2_probability.json")
    
    # 2. 检查目录是否存在
    if not os.path.exists(agg_dir):
        print(f"[Warn] aggregated_results 不存在: {agg_dir}")
        return pd.DataFrame()

    # 3. 加载 Ground Truth 白名单
    valid_queries = set()
    if os.path.exists(gt_path):
        try:
            with open(gt_path, 'r') as f:
                gt_data = json.load(f)
                # JSON keys 是带后缀的文件名，如 "query_cycle_10_144.graph"
                valid_queries = set(gt_data.keys())
            print(f"[{dataset_name}] 加载 Ground Truth 白名单成功，共 {len(valid_queries)} 个有效查询。")
        except Exception as e:
            print(f"[Error] 读取 Ground Truth JSON 失败: {e}")
            return pd.DataFrame()
    else:
        print(f"[Warn] Ground Truth 文件不存在: {gt_path}，跳过该数据集。")
        return pd.DataFrame()

    records = []
    
    # 增加容错，防止目录为空报错
    try:
        file_list = sorted(os.listdir(agg_dir))
    except Exception as e:
        print(f"[Error] 无法列出目录 {agg_dir}: {e}")
        return pd.DataFrame()
    
    print(f"正在处理数据集: {dataset_name}, 扫描 aggregated_results 目录...")

    for fname in file_list:
        if not fname.endswith(".csv"):
            continue
            
        # --- [核心过滤逻辑] ---
        # 提取 query_basename (例如: aggregated_list_q1.csv -> q1.graph)
        base = fname.replace("aggregated_list_", "").replace(".csv", "")
        query_basename = f"{base}.graph"
        
        # 检查是否在白名单中
        if query_basename not in valid_queries:
            # print(f"[Skip] {query_basename} 不在 Ground Truth 中") # 可选：打印跳过的文件
            continue
        # --------------------

        fpath = os.path.join(agg_dir, fname)

        try:
            # 1. 先读表头，判断列名，只读取需要的列以节省内存
            header = pd.read_csv(fpath, nrows=0)
            if "estimateW" in header.columns:
                col = "estimateW"
            elif "a" in header.columns:
                col = "a"
            else:
                continue

            # 2. 读取数据列
            df = pd.read_csv(fpath, usecols=[col])
            
            # 3. 处理数据 (转数字，填充NaN，统计 >0 的行)
            vals = pd.to_numeric(df[col], errors="coerce").fillna(0.0)
            core_cnt = int((vals > 0).sum())
            total_rows = int(len(vals))

            records.append({
                "dataset_name": dataset_name,
                "query_basename": query_basename,
                "total_rows": total_rows,
                "core_instances": core_cnt
            })
        except Exception as e:
            print(f"[Error] 读取文件 {fname} 出错: {e}")

    print(f"[{dataset_name}] 处理完成，有效记录数: {len(records)}")
    return pd.DataFrame(records)

def plot_distributions(df_all, output_dir="."):
    """
    为每个数据集绘制 count 分布图 (直方图) 和 累计分布图 (CDF)
    """
    # 设置绘图风格
    sns.set(style="whitegrid", context="notebook") 
    
    datasets = df_all['dataset_name'].unique()
    
    for ds in datasets:
        subset = df_all[df_all['dataset_name'] == ds]
        
        if subset.empty:
            continue
            
        # 创建画布：1行2列
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        fig.suptitle(f"Core Instances Analysis - {ds}", fontsize=16)
        
        # --- 图1: 直方图 (Histogram + KDE) ---
        sns.histplot(data=subset, x="core_instances", kde=True, bins=30, color="skyblue", ax=axes[0])
        
        mean_val = subset['core_instances'].mean()
        median_val = subset['core_instances'].median()
        
        axes[0].axvline(mean_val, color='r', linestyle='--', label=f'Mean: {mean_val:.1f}')
        axes[0].axvline(median_val, color='g', linestyle='-', label=f'Median: {median_val:.1f}')
        axes[0].set_title("Frequency Distribution (Histogram)")
        axes[0].set_xlabel("Count of Core Instances")
        axes[0].set_ylabel("Count of Queries")
        axes[0].legend()

        # --- 图2: 累计分布图 (CDF) ---
        sns.ecdfplot(data=subset, x="core_instances", ax=axes[1], linewidth=2, color="purple")
        
        p95_val = subset['core_instances'].quantile(0.95)
        axes[1].axvline(p95_val, color='orange', linestyle=':', linewidth=2, label=f'P95: {p95_val:.1f}')
        axes[1].axhline(0.95, color='orange', linestyle=':', linewidth=1)
        
        axes[1].set_title("Cumulative Distribution Function (CDF)")
        axes[1].set_xlabel("Count of Core Instances")
        axes[1].set_ylabel("Proportion")
        axes[1].grid(True, which="both", linestyle="--", alpha=0.7)
        axes[1].legend()

        plt.tight_layout()
        save_path = os.path.join(output_dir, f"{ds}_distribution_cdf.png")
        plt.savefig(save_path)
        print(f"[Plot] 已保存图表: {save_path}")
        
        plt.show()

# ==========================================
# 主执行流程
# ==========================================
# if __name__ == "__main__":
#     # 配置数据集列表
#     datasets = ["dataset_test", "dataset_test2", "dataset_test3", "dataset_three"]
#     # 基础路径
#     base_data_dir = "/home/wangshuo/resource/datasets/parler_data"
    
#     print("=== 开始统计 ===")
    
#     # 1. 收集所有数据
#     df_list = []
#     for ds in datasets:
#         df_ds = count_core_instances(ds, base_dir=base_data_dir)
#         if not df_ds.empty:
#             df_list.append(df_ds)
    
#     if df_list:
#         df_all = pd.concat(df_list, ignore_index=True)
        
#         # 2. 控制台输出统计摘要
#         print("\n" + "="*60)
#         print("统计结果预览 (前10行):")
#         print(df_all.head(10))
        
#         print("\n" + "="*60)
#         print("各数据集统计摘要 (Describe):")
#         desc = df_all.groupby("dataset_name")["core_instances"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
#         pd.set_option('display.max_columns', None) 
#         pd.set_option('display.width', 1000)
#         print(desc)
#         print("="*60)

#         # 3. 保存到 CSV
#         csv_filename = "D_count.csv"
#         df_all.to_csv(csv_filename, index=False)
#         print(f"\n[Save] 完整统计已保存至: {os.path.abspath(csv_filename)}")

#         # 4. 绘制分布图
#         print("\n[Plot] 正在生成分布图并显示...")
#         plot_distributions(df_all)
        
#     else:
#         print("[Error] 未获取到任何数据，请检查路径配置。")

统计各数据集的核心实例数, 保存到D_count_{dataset_name}.csv文件中去

In [5]:
import os
import pandas as pd

def save_core_instance_counts_per_dataset(
    datasets,
    base_data_dir="/home/wangshuo/resource/datasets/parler_data",
    out_dir="."
):
    os.makedirs(out_dir, exist_ok=True)

    for ds in datasets:
        df_ds = count_core_instances(ds, base_dir=base_data_dir)
        if df_ds.empty:
            print(f"[Skip] {ds}: no valid records.")
            continue

        out_path = os.path.join(out_dir, f"D_count_{ds}.csv")
        df_ds.to_csv(out_path, index=False)
        print(f"[Save] {ds}: {len(df_ds)} queries -> {out_path}")

# 你要统计的数据集列表
datasets = ["dataset_test", "dataset_test2", "dataset_test3", "dataset_three"]

# 保存位置：默认当前目录（你也可以改成某个 results 目录）
save_core_instance_counts_per_dataset(
    datasets=datasets,
    base_data_dir="/home/wangshuo/resource/datasets/parler_data",
    out_dir="."
)

[dataset_test] 加载 Ground Truth 白名单成功，共 113 个有效查询。
正在处理数据集: dataset_test, 扫描 aggregated_results 目录...
[dataset_test] 处理完成，有效记录数: 113
[Save] dataset_test: 113 queries -> ./D_count_dataset_test.csv
[dataset_test2] 加载 Ground Truth 白名单成功，共 290 个有效查询。
正在处理数据集: dataset_test2, 扫描 aggregated_results 目录...
[dataset_test2] 处理完成，有效记录数: 290
[Save] dataset_test2: 290 queries -> ./D_count_dataset_test2.csv
[dataset_test3] 加载 Ground Truth 白名单成功，共 174 个有效查询。
正在处理数据集: dataset_test3, 扫描 aggregated_results 目录...
[dataset_test3] 处理完成，有效记录数: 174
[Save] dataset_test3: 174 queries -> ./D_count_dataset_test3.csv
[dataset_three] 加载 Ground Truth 白名单成功，共 246 个有效查询。
正在处理数据集: dataset_three, 扫描 aggregated_results 目录...
[dataset_three] 处理完成，有效记录数: 246
[Save] dataset_three: 246 queries -> ./D_count_dataset_three.csv


In [3]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json

def count_core_instances(dataset_name: str,
                         base_dir: str = "/home/wangshuo/resource/datasets/parler_data"):
    """
    统计每个查询的核心实例个数：
    核心实例 = aggregated_results 中 estimateW > 0 (或 a > 0) 的行数
    【过滤条件】：只统计在 ground_truth JSON 文件中存在的查询。
    """
    # 1. 路径定义
    agg_dir = os.path.join(base_dir, dataset_name, "results", "aggregated_results")
    gt_path = os.path.join(base_dir, dataset_name, "results", "T_true_ML1_oracle2_probability_ML2_oracle2_probability.json")
    
    # 2. 检查目录是否存在
    if not os.path.exists(agg_dir):
        print(f"[Warn] aggregated_results 不存在: {agg_dir}")
        return pd.DataFrame()

    # 3. 加载 Ground Truth 白名单
    valid_queries = set()
    if os.path.exists(gt_path):
        try:
            with open(gt_path, 'r') as f:
                gt_data = json.load(f)
                valid_queries = set(gt_data.keys())
            print(f"[{dataset_name}] 加载 Ground Truth 白名单成功，共 {len(valid_queries)} 个有效查询。")
        except Exception as e:
            print(f"[Error] 读取 Ground Truth JSON 失败: {e}")
            return pd.DataFrame()
    else:
        print(f"[Warn] Ground Truth 文件不存在: {gt_path}，跳过该数据集。")
        return pd.DataFrame()

    records = []
    
    try:
        file_list = sorted(os.listdir(agg_dir))
    except Exception as e:
        print(f"[Error] 无法列出目录 {agg_dir}: {e}")
        return pd.DataFrame()
    
    print(f"正在处理数据集: {dataset_name}, 扫描 aggregated_results 目录...")

    for fname in file_list:
        if not fname.endswith(".csv"):
            continue
            
        # --- [核心过滤逻辑] ---
        base = fname.replace("aggregated_list_", "").replace(".csv", "")
        query_basename = f"{base}.graph"
        
        if query_basename not in valid_queries:
            continue
        # --------------------

        fpath = os.path.join(agg_dir, fname)

        try:
            header = pd.read_csv(fpath, nrows=0)
            if "estimateW" in header.columns:
                col = "estimateW"
            elif "a" in header.columns:
                col = "a"
            else:
                continue

            # 读取数据列
            df = pd.read_csv(fpath, usecols=[col])
            
            # 统计 >0 的行
            vals = pd.to_numeric(df[col], errors="coerce").fillna(0.0)
            core_cnt = int((vals > 0).sum())
            total_rows = int(len(vals))

            records.append({
                "dataset_name": dataset_name,
                "query_basename": query_basename,
                "total_rows": total_rows,
                "core_instances": core_cnt
            })
        except Exception as e:
            print(f"[Error] 读取文件 {fname} 出错: {e}")

    print(f"[{dataset_name}] 处理完成，有效记录数: {len(records)}")
    return pd.DataFrame(records)

def plot_distributions(df_all, output_dir="."):
    """
    为每个数据集绘制 count 分布图 (直方图) 和 累计分布图 (CDF)
    """
    sns.set(style="whitegrid", context="notebook") 
    
    datasets = df_all['dataset_name'].unique()
    
    for ds in datasets:
        subset = df_all[df_all['dataset_name'] == ds]
        
        if subset.empty:
            continue
            
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        fig.suptitle(f"Core Instances Analysis - {ds}", fontsize=16)
        
        # --- 图1: 直方图 ---
        sns.histplot(data=subset, x="core_instances", kde=True, bins=30, color="skyblue", ax=axes[0])
        
        mean_val = subset['core_instances'].mean()
        median_val = subset['core_instances'].median()
        
        axes[0].axvline(mean_val, color='r', linestyle='--', label=f'Mean: {mean_val:.1f}')
        axes[0].axvline(median_val, color='g', linestyle='-', label=f'Median: {median_val:.1f}')
        axes[0].set_title("Frequency Distribution (Histogram)")
        axes[0].set_xlabel("Count of Core Instances")
        axes[0].set_ylabel("Count of Queries")
        axes[0].legend()

        # --- 图2: CDF ---
        sns.ecdfplot(data=subset, x="core_instances", ax=axes[1], linewidth=2, color="purple")
        
        p95_val = subset['core_instances'].quantile(0.95)
        axes[1].axvline(p95_val, color='orange', linestyle=':', linewidth=2, label=f'P95: {p95_val:.1f}')
        axes[1].axhline(0.95, color='orange', linestyle=':', linewidth=1)
        
        axes[1].set_title("Cumulative Distribution Function (CDF)")
        axes[1].set_xlabel("Count of Core Instances")
        axes[1].set_ylabel("Proportion")
        axes[1].grid(True, which="both", linestyle="--", alpha=0.7)
        axes[1].legend()

        plt.tight_layout()
        save_path = os.path.join(output_dir, f"{ds}_distribution_cdf.png")
        plt.savefig(save_path)
        print(f"[Plot] 已保存图表: {save_path}")
        
        # plt.show() # 如果在无界面环境运行，可以注释掉

# ==========================================
# 主执行流程
# ==========================================
if __name__ == "__main__":
    # 配置数据集列表
    datasets = ["dataset_test", "dataset_test2", "dataset_test3", "dataset_three"]
    # 基础路径
    base_data_dir = "/home/wangshuo/resource/datasets/parler_data"
    
    print("=== 开始统计 ===")
    
    # 1. 收集所有数据
    df_list = []
    for ds in datasets:
        df_ds = count_core_instances(ds, base_dir=base_data_dir)
        if not df_ds.empty:
            df_list.append(df_ds)
    
    if df_list:
        df_all = pd.concat(df_list, ignore_index=True)
        
        # 2. 控制台输出常规统计摘要
        print("\n" + "="*60)
        print("各数据集常规统计摘要 (Describe):")
        desc = df_all.groupby("dataset_name")["core_instances"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
        pd.set_option('display.max_columns', None) 
        pd.set_option('display.width', 1000)
        print(desc)
        
        # =======================================================
        # [新增] 统计核心集 < 100 的个数和比例
        # =======================================================
        print("\n" + "="*60)
        print("【重点关注】核心实例 < 100 的查询统计:")
        print("-" * 60)
        ctn = 50
        # 使用 groupby 和 apply 计算
        small_core_stats = df_all.groupby('dataset_name').apply(
            lambda x: pd.Series({
                'Total Queries': len(x),
                'Count < 100': (x['core_instances'] < ctn).sum(),
                'Ratio < 100 (%)': ((x['core_instances'] < ctn).sum() / len(x)) * 100
            })
        ).reset_index()
        
        # 格式化输出
        print(f"{'Dataset Name':<20} | {'Total':<10} | {'Count < 100':<12} | {'Ratio (%)':<10}")
        print("-" * 60)
        for _, row in small_core_stats.iterrows():
            print(f"{row['dataset_name']:<20} | {int(row['Total Queries']):<10} | "
                  f"{int(row['Count < 100']):<12} | {row['Ratio < 100 (%)']:.2f}%")
        print("="*60)
        # =======================================================

        # 3. 保存到 CSV
        csv_filename = "D_count.csv"
        # 这里为了后续使用方便，我们只保存原始数据，不保存汇总数据
        df_all.to_csv(csv_filename, index=False)
        print(f"\n[Save] 完整明细数据已保存至: {os.path.abspath(csv_filename)}")

        # 4. 绘制分布图
        # print("\n[Plot] 正在生成分布图...")
        # plot_distributions(df_all)
        
    else:
        print("[Error] 未获取到任何数据，请检查路径配置。")

=== 开始统计 ===
[dataset_test] 加载 Ground Truth 白名单成功，共 113 个有效查询。
正在处理数据集: dataset_test, 扫描 aggregated_results 目录...
[dataset_test] 处理完成，有效记录数: 113
[dataset_test2] 加载 Ground Truth 白名单成功，共 290 个有效查询。
正在处理数据集: dataset_test2, 扫描 aggregated_results 目录...
[dataset_test2] 处理完成，有效记录数: 290
[dataset_test3] 加载 Ground Truth 白名单成功，共 174 个有效查询。
正在处理数据集: dataset_test3, 扫描 aggregated_results 目录...
[dataset_test3] 处理完成，有效记录数: 174
[dataset_three] 加载 Ground Truth 白名单成功，共 246 个有效查询。
正在处理数据集: dataset_three, 扫描 aggregated_results 目录...
[dataset_three] 处理完成，有效记录数: 246

各数据集常规统计摘要 (Describe):
               count         mean          std    min      25%     50%      75%     90%      95%      99%      max
dataset_name                                                                                                      
dataset_test   113.0  4828.619469  2605.298281  664.0  2461.00  4925.0  6502.00  9081.6  9328.40  9519.28   9621.0
dataset_test2  290.0  1550.265517  1262.093716   32.0   486.25  1379.5  2315.25  

#### 2. FOIS_nrs和FOIS_rs在每个查询不同采样率下的结果保存到了某数据集下的FOIS_rs_POSS_budget_curve.csv文件中, 而且对于每个查询,每个采样率都运行了5次(取绝对值误差平均值作为该采样率下该查询的估计误差标准), 我想让你统计给定数据集和采样率后,哪些查询的误差标准FOIS_nrs 小于 FOIS_rs ,同时给出这些查询的核心实例个数(就是/home/wangshuo/resource/datasets/parler_data/{dataset_name}/results/aggregated_results文件夹下各文件的元组数量(满足列estimateW > 0)),输出打印出来相关信息并保存

In [ ]:
import os
import json
import pandas as pd
import numpy as np

def _load_gt_whitelist(dataset_name: str,
                       post_oracle_col="ML1_oracle2_probability",
                       comment_oracle_col="ML2_oracle2_probability",
                       base_dir="/home/wangshuo/resource/datasets/parler_data"):
    """
    读取 GroundTruth JSON，用于过滤有效查询（只统计 GT 里出现的 query）。
    """
    gt_path = os.path.join(
        base_dir,
        dataset_name,
        "ground_truth",
        f"T_true_{post_oracle_col}_{comment_oracle_col}.json"
    )
    if not os.path.exists(gt_path):
        # 兼容你 notebook 里旧路径（如果你那边是 results 下的 json）
        alt = os.path.join(
            base_dir,
            dataset_name,
            "results",
            f"T_true_{post_oracle_col}_{comment_oracle_col}.json"
        )
        gt_path = alt

    if not os.path.exists(gt_path):
        print(f"[Warn] GT json not found, skip whitelist filter: {gt_path}")
        return None

    with open(gt_path, "r") as f:
        gt = json.load(f)
    return set(gt.keys())


def count_core_instances_for_queries(dataset_name: str,
                                    queries: list,
                                    base_dir="/home/wangshuo/resource/datasets/parler_data"):
    """
    给定 query_basename 列表，统计每个查询对应 aggregated_list_*.csv 里 estimateW>0 的行数。
    返回 df: query_basename, core_instances, total_rows
    """
    agg_dir = os.path.join(base_dir, dataset_name, "results", "aggregated_results")
    if not os.path.exists(agg_dir):
        raise FileNotFoundError(f"aggregated_results not found: {agg_dir}")

    query_set = set(queries)
    records = []

    # 建立 filename -> query_basename 的映射规则，与 run_budget_curve_multi_predicate 一致
    for fname in sorted(os.listdir(agg_dir)):
        if not fname.endswith(".csv"):
            continue

        base = fname.replace("aggregated_list_", "").replace(".csv", "")
        query_basename = f"{base}.graph"
        if query_basename not in query_set:
            continue

        fpath = os.path.join(agg_dir, fname)

        header = pd.read_csv(fpath, nrows=0)
        if "estimateW" in header.columns:
            col = "estimateW"
        elif "a" in header.columns:
            col = "a"
        else:
            print(f"[Warn] {fname} missing estimateW/a, skip")
            continue

        df = pd.read_csv(fpath, usecols=[col])
        vals = pd.to_numeric(df[col], errors="coerce").fillna(0.0)
        core_cnt = int((vals > 0).sum())

        records.append({
            "query_basename": query_basename,
            "core_instances": core_cnt,
            "total_rows": int(len(vals)),
            "aggregated_file": fname
        })

    return pd.DataFrame(records)


def analyse_nrs_better_than_rs(dataset_name: str,
                               budget_frac: float,
                               post_oracle_col="ML1_oracle2_probability",
                               comment_oracle_col="ML2_oracle2_probability",
                               base_dir="/home/wangshuo/resource/datasets/parler_data",
                               out_dir=None):
    """
    统计：在给定 dataset + budget_frac 下，哪些 query 的 mean(Qerror) 满足
    FOIS_nrs < FOIS_rs，并输出其 core_instances。
    """
    # 1) 读取 FOIS 预算曲线结果
    curve_path = os.path.join(base_dir, dataset_name, "results", "efficiency", "FOIS_rs_POSS_budget_curve.csv")
    if not os.path.exists(curve_path):
        # 有些数据放在 budget_curve 目录
        alt = os.path.join(base_dir, dataset_name, "results", "budget_curve", "FOIS_rs_POSS_budget_curve.csv")
        if os.path.exists(alt):
            curve_path = alt
        else:
            raise FileNotFoundError(f"FOIS curve csv not found: {curve_path}")

    df = pd.read_csv(curve_path)

    # 统一 query_basename 形式
    if "query_basename" not in df.columns and "query_name" in df.columns:
        df.rename(columns={"query_name": "query_basename"}, inplace=True)
    df["query_basename"] = df["query_basename"].astype(str)
    df["query_basename"] = df["query_basename"].apply(lambda x: x if x.endswith(".graph") else f"{x}.graph")

    # 2) 只取该 budget_frac 下的数据（浮点做 isclose）
    df["budget_frac"] = pd.to_numeric(df["budget_frac"], errors="coerce")
    df_sub = df[np.isclose(df["budget_frac"].values, budget_frac, atol=1e-9)].copy()
    if df_sub.empty:
        print(f"[Warn] No rows for budget_frac={budget_frac} in {curve_path}")
        return None

    # 3) 可选：只统计 GT 白名单里的 query
    whitelist = _load_gt_whitelist(dataset_name, post_oracle_col, comment_oracle_col, base_dir=base_dir)
    if whitelist is not None:
        df_sub = df_sub[df_sub["query_basename"].isin(whitelist)].copy()

    # 4) 对每个 query + method 计算 mean Qerror（你说的“误差标准”）
    df_sub["Qerror"] = pd.to_numeric(df_sub["Qerror"], errors="coerce")
    df_agg = (
        df_sub[df_sub["method"].isin(["FOIS_nrs", "FOIS_rs"])]
        .groupby(["query_basename", "method"])["Qerror"]
        .mean()
        .reset_index()
        .pivot(index="query_basename", columns="method", values="Qerror")
        .reset_index()
    )

    # 有些 query 可能缺某个方法，过滤掉
    df_agg = df_agg.dropna(subset=["FOIS_nrs", "FOIS_rs"]).copy()

    df_agg["nrs_minus_rs"] = df_agg["FOIS_nrs"] - df_agg["FOIS_rs"]
    df_better = df_agg[df_agg["FOIS_nrs"] < df_agg["FOIS_rs"]].copy()
    df_better = df_better.sort_values("nrs_minus_rs")  # 越负表示 nrs 优势越大

    if df_better.empty:
        print(f"[Result] No queries where FOIS_nrs < FOIS_rs at budget_frac={budget_frac}.")
        return df_better

    # 5) 统计这些 query 的 core_instances
    df_core = count_core_instances_for_queries(dataset_name, df_better["query_basename"].tolist(), base_dir=base_dir)
    df_final = df_better.merge(df_core, on="query_basename", how="left")

    # 6) 打印信息
    print("=" * 80)
    print(f"[Dataset] {dataset_name}")
    print(f"[Budget] budget_frac = {budget_frac}")
    print(f"[Count] FOIS_nrs < FOIS_rs queries = {len(df_final)}")
    print("-" * 80)
    display_cols = ["query_basename", "FOIS_nrs", "FOIS_rs", "nrs_minus_rs", "core_instances", "total_rows", "aggregated_file"]
    print(df_final[display_cols].head(30).to_string(index=False))
    print("=" * 80)

    # 7) 保存
    if out_dir is None:
        out_dir = os.path.join(base_dir, dataset_name, "results", "efficiency")
    os.makedirs(out_dir, exist_ok=True)

    out_path = os.path.join(out_dir, f"nrs_better_than_rs_budget_{budget_frac:.2f}.csv")
    df_final.to_csv(out_path, index=False)
    print(f"[Save] {out_path}")

    return df_final


# =========================
# 用法示例（改 dataset/budget）
# =========================
dataset_name = "dataset_test3"
budget_frac = 0.10  # 例如 0.05 / 0.10 / 0.15 / ... / 0.5

df_result = analyse_nrs_better_than_rs(
    dataset_name=dataset_name,
    budget_frac=budget_frac,
    post_oracle_col="ML1_oracle2_probability",
    comment_oracle_col="ML2_oracle2_probability",
)
df_result.head()

#### 2.1 绘图 ,FOIS准确率和核心集大小的关系:

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 设置绘图风格
sns.set(style="whitegrid")

def _load_gt_whitelist(dataset_name: str,
                       post_oracle_col="ML1_oracle2_probability",
                       comment_oracle_col="ML2_oracle2_probability",
                       base_dir="/home/wangshuo/resource/datasets/parler_data"):
    """
    读取 GroundTruth JSON，用于过滤有效查询（只统计 GT 里出现的 query）。
    """
    gt_path = os.path.join(
        base_dir,
        dataset_name,
        "ground_truth",
        f"T_true_{post_oracle_col}_{comment_oracle_col}.json"
    )
    # 兼容备用路径
    if not os.path.exists(gt_path):
        alt = os.path.join(base_dir, dataset_name, "results", f"T_true_{post_oracle_col}_{comment_oracle_col}.json")
        gt_path = alt

    if not os.path.exists(gt_path):
        print(f"[Warn] GT json not found, skip whitelist filter: {gt_path}")
        return None

    with open(gt_path, "r") as f:
        gt = json.load(f)
    return set(gt.keys())


def count_core_instances_for_queries(dataset_name: str,
                                     queries: list,
                                     base_dir="/home/wangshuo/resource/datasets/parler_data"):
    """
    给定 query_basename 列表，统计每个查询对应 aggregated_list_*.csv 里 estimateW > 0 的行数。
    """
    agg_dir = os.path.join(base_dir, dataset_name, "results", "aggregated_results")
    if not os.path.exists(agg_dir):
        print(f"[Warn] aggregated_results dir not found: {agg_dir}")
        # 如果找不到文件夹，返回全0
        return pd.DataFrame({"query_basename": queries, "core_instances": 0, "total_rows": 0})

    query_set = set(queries)
    records = []
    
    # 遍历目录下所有文件
    # 文件名通常格式: aggregated_list_{query_id}.csv
    # 对应的 query_basename: {query_id}.graph
    
    files = [f for f in os.listdir(agg_dir) if f.endswith(".csv")]
    
    # 建立映射缓存以加速查找，避免每次循环都遍历所有文件
    file_map = {}
    for fname in files:
        base = fname.replace("aggregated_list_", "").replace(".csv", "")
        q_name = f"{base}.graph"
        file_map[q_name] = fname

    for q in queries:
        # 确保 q 是 .graph 结尾
        q_name = q if q.endswith(".graph") else f"{q}.graph"
        
        fname = file_map.get(q_name)
        if not fname:
            # 可能是没找到对应文件
            records.append({
                "query_basename": q_name,
                "core_instances": 0,
                "total_rows": 0,
                "aggregated_file": None
            })
            continue

        fpath = os.path.join(agg_dir, fname)
        try:
            # 只读取头部判断列名
            header = pd.read_csv(fpath, nrows=0)
            if "estimateW" in header.columns:
                col = "estimateW"
            elif "a" in header.columns:
                col = "a"
            else:
                col = None
            
            if col:
                df_csv = pd.read_csv(fpath, usecols=[col])
                vals = pd.to_numeric(df_csv[col], errors="coerce").fillna(0.0)
                core_cnt = int((vals > 0).sum())
                total = int(len(vals))
            else:
                core_cnt = 0
                total = 0
        except Exception as e:
            print(f"Error reading {fname}: {e}")
            core_cnt = 0
            total = 0

        records.append({
            "query_basename": q_name,
            "core_instances": core_cnt,
            "total_rows": total,
            "aggregated_file": fname
        })

    return pd.DataFrame(records)


def visualize_analysis(df_merged, dataset_name, budget_frac, out_dir):
    """
    可视化：
    1. 散点图：Core Instances vs Error Difference
    2. CDF图：Better组 vs Worse组 的 Core Instances 分布
    """
    if df_merged.empty:
        return

    # 添加分组标签
    df_merged["Group"] = df_merged.apply(
        lambda x: "NRS Better" if x["nrs_minus_rs"] < 0 else "NRS Worse/Equal", axis=1
    )

    # 1. 散点图 (Scatter Plot)
    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        data=df_merged, 
        x="core_instances", 
        y="nrs_minus_rs", 
        hue="Group", 
        palette={"NRS Better": "green", "NRS Worse/Equal": "red"},
        alpha=0.6
    )
    plt.axhline(0, color='black', linestyle='--', linewidth=1)
    plt.title(f"[{dataset_name} | Budget {budget_frac}] Error Diff vs Core Instances")
    plt.xlabel("Core Instances (Count of estimateW > 0)")
    plt.ylabel("Diff (FOIS_nrs - FOIS_rs)\n< 0 means NRS is better")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"scatter_diff_vs_core_{budget_frac}.png"))
    plt.show()
    plt.close()

    # 2. CDF 图 (Cumulative Distribution Function)
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(
        data=df_merged, 
        x="core_instances", 
        hue="Group", 
        palette={"NRS Better": "green", "NRS Worse/Equal": "red"},
        linewidth=2
    )
    plt.title(f"[{dataset_name} | Budget {budget_frac}] CDF of Core Instances")
    plt.xlabel("Core Instances")
    plt.ylabel("Proportion of Queries")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"cdf_core_instances_{budget_frac}.png"))
    plt.show()
    plt.close()
    print(f"[Vis] Plots saved to {out_dir}")


def analyze_nrs_vs_rs_performance(dataset_name: str,
                                  budget_frac: float,
                                  post_oracle_col="ML1_oracle2_probability",
                                  comment_oracle_col="ML2_oracle2_probability",
                                  base_dir="/home/wangshuo/resource/datasets/parler_data"):
    
    # 1. 确定输出目录
    out_dir = os.path.join(base_dir, dataset_name, "results", "analysis_nrs_vs_rs")
    os.makedirs(out_dir, exist_ok=True)

    # 2. 读取 Budget Curve CSV
    # 优先查找 efficienty 目录，其次 budget_curve 目录
    paths_to_check = [
        os.path.join(base_dir, dataset_name, "results", "efficiency", "FOIS_rs_POSS_budget_curve.csv"),
        os.path.join(base_dir, dataset_name, "results", "budget_curve", "FOIS_rs_POSS_budget_curve.csv")
    ]
    curve_path = None
    for p in paths_to_check:
        if os.path.exists(p):
            curve_path = p
            break
    
    if not curve_path:
        print(f"[Error] Budget curve CSV not found in checked paths.")
        return

    print(f"[Load] Reading {curve_path}...")
    df = pd.read_csv(curve_path)

    # 标准化列名
    if "query_name" in df.columns:
        df.rename(columns={"query_name": "query_basename"}, inplace=True)
    
    # 过滤采样率 (budget_frac)
    df["budget_frac"] = pd.to_numeric(df["budget_frac"], errors="coerce")
    df_sub = df[np.isclose(df["budget_frac"], budget_frac, atol=1e-9)].copy()
    
    if df_sub.empty:
        print(f"[Warn] No data found for budget_frac={budget_frac}")
        return

    # 3. 过滤白名单 (GT)
    whitelist = _load_gt_whitelist(dataset_name, post_oracle_col, comment_oracle_col, base_dir)
    if whitelist:
        # 确保格式一致（都是 .graph 结尾）
        df_sub["query_basename"] = df_sub["query_basename"].astype(str).apply(
            lambda x: x if x.endswith(".graph") else f"{x}.graph"
        )
        original_count = len(df_sub["query_basename"].unique())
        df_sub = df_sub[df_sub["query_basename"].isin(whitelist)]
        print(f"[Filter] Whitelist applied: {original_count} -> {len(df_sub['query_basename'].unique())} queries")

    # 4. 计算平均误差 (Run 5次的平均值)
    # 假设列名是 Qerror (绝对误差)
    if "Qerror" not in df_sub.columns:
        # 如果 csv 里叫 abs_error 之类的，可以在这里改
        if "abs_error" in df_sub.columns:
            df_sub.rename(columns={"abs_error": "Qerror"}, inplace=True)
        else:
            print("[Error] No 'Qerror' column found.")
            return

    target_methods = ["FOIS_nrs", "FOIS_rs"]
    df_agg = (
        df_sub[df_sub["method"].isin(target_methods)]
        .groupby(["query_basename", "method"])["Qerror"]
        .mean()
        .reset_index()
    )

    # 透视表: query | FOIS_nrs | FOIS_rs
    df_pivot = df_agg.pivot(index="query_basename", columns="method", values="Qerror").reset_index()
    
    # 移除缺失任一方法的查询
    df_pivot.dropna(subset=target_methods, inplace=True)

    # 计算差值
    df_pivot["nrs_minus_rs"] = df_pivot["FOIS_nrs"] - df_pivot["FOIS_rs"]

    # 5. 获取 Core Instances
    print(f"[Analysis] Counting core instances for {len(df_pivot)} queries...")
    df_core = count_core_instances_for_queries(dataset_name, df_pivot["query_basename"].tolist(), base_dir)
    
    # 合并
    df_final = df_pivot.merge(df_core, on="query_basename", how="left")
    
    # 6. 拆分 Better / Worse
    df_better = df_final[df_final["nrs_minus_rs"] < 0].sort_values("nrs_minus_rs")
    df_worse = df_final[df_final["nrs_minus_rs"] >= 0].sort_values("nrs_minus_rs", ascending=False)

    # 7. 打印并保存
    cols = ["query_basename", "FOIS_nrs", "FOIS_rs", "nrs_minus_rs", "core_instances", "total_rows"]
    
    print("\n" + "="*80)
    print(f" Dataset: {dataset_name} | Budget: {budget_frac}")
    print(f" Queries where FOIS_nrs BETTER than FOIS_rs (nrs < rs): {len(df_better)}")
    print("-" * 80)
    print(df_better[cols].head(10).to_string(index=False))
    
    print("\n" + "-" * 80)
    print(f" Queries where FOIS_nrs WORSER/EQUAL to FOIS_rs (nrs >= rs): {len(df_worse)}")
    print("-" * 80)
    print(df_worse[cols].head(10).to_string(index=False))
    print("="*80 + "\n")

    # 保存文件
    path_better = os.path.join(out_dir, f"nrs_better_than_rs_budget_{budget_frac}.csv")
    path_worse = os.path.join(out_dir, f"nrs_worser_than_rs_budget_{budget_frac}.csv")
    path_all = os.path.join(out_dir, f"all_nrs_vs_rs_comparison_budget_{budget_frac}.csv")

    df_better.to_csv(path_better, index=False)
    df_worse.to_csv(path_worse, index=False)
    df_final.to_csv(path_all, index=False)

    print(f"[Save] Better: {path_better}")
    print(f"[Save] Worse:  {path_worse}")

    # 8. 可视化
    visualize_analysis(df_final, dataset_name, budget_frac, out_dir)


if __name__ == "__main__":
    # 配置区
    DATASET_NAME = "dataset_test"   # 请修改为实际数据集名称
    BUDGET_FRAC = 0.10              # 请修改为实际采样率
    
    analyze_nrs_vs_rs_performance(
        dataset_name=DATASET_NAME,
        budget_frac=BUDGET_FRAC
    )

#### 2.2 上面代码是FOIS准确率和核心集大小的关系,你应该知道aggregated_list_query_cycle_4_0.csv 等文件和FOIS_rs_POSS_budget_curve.csv 文件的含义和部分内容吧,我想弄清FOIS_nrs和FOIS_rs的误差率和查询的什么信息有关,方便我根据查询的特定特征选择合适的采样方法(放回还是不放回), 我们已知每个查询各个核心集的estimateW 和 核心集点相关的代理分数,这些都保存到了aggregated_list_query_cycle_4_0.csv 等文件(对应查询的核心集文件中),模仿FOIS准确率和核心集X大小的关系代码, 分别统计 FOIS_nrs和FOIS_rs的误差率和 global_estimateW (联合结构权重global_estimateW = sigma(W(X))), 平均联合代理分数p(X)(核心集多个节点代理分数乘积平均值 P(X) = p1(x1)*p2(x2)....),联合重要性权重( w(X)*P(X))的关系

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast

# 设置绘图风格
sns.set(style="whitegrid")

def _load_gt_whitelist(dataset_name: str,
                       post_oracle_col="ML1_oracle2_probability",
                       comment_oracle_col="ML2_oracle2_probability",
                       base_dir="/home/wangshuo/resource/datasets/parler_data"):
    # ... (保持不变) ...
    gt_path = os.path.join(
        base_dir, dataset_name, "ground_truth",
        f"T_true_{post_oracle_col}_{comment_oracle_col}.json"
    )
    if not os.path.exists(gt_path):
        alt = os.path.join(base_dir, dataset_name, "results", f"T_true_{post_oracle_col}_{comment_oracle_col}.json")
        gt_path = alt
    if not os.path.exists(gt_path):
        # 尝试默认路径
        gt_path = os.path.join(base_dir, dataset_name, "results", "T_true.json")
    
    if not os.path.exists(gt_path):
        print(f"[Warn] GT json not found, skip whitelist filter: {gt_path}")
        return None

    with open(gt_path, "r") as f:
        gt = json.load(f)
    return set(gt.keys())

def safe_literal_eval(val):
    if pd.isna(val) or not isinstance(val, str) or val == 'nan':
        return []
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return []

def calculate_joint_proxy(row, post_col="ML1_proxy4b1_probability", comment_col="ML2_proxy1_probability"):
    """计算单个实例的联合代理分数"""
    p_list = safe_literal_eval(row.get(post_col))
    c_list = safe_literal_eval(row.get(comment_col))
    
    # 将列表转换为数值并计算乘积
    p_prod = np.nanprod([pd.to_numeric(x, errors='coerce') for x in p_list])
    c_prod = np.nanprod([pd.to_numeric(x, errors='coerce') for x in c_list])
    
    # 联合概率 = Post乘积 * Comment乘积
    return p_prod * c_prod

def extract_features_for_queries(dataset_name: str,
                                 queries: list,
                                 base_dir="/home/wangshuo/resource/datasets/parler_data"):
    """
    给定 query_basename 列表，从 aggregated_list_*.csv 中提取高级特征：
    1. core_instances: estimateW > 0 的行数
    2. global_estimateW: sum(estimateW)
    3. avg_joint_proxy: mean(joint_proxy)
    4. total_importance_weight: sum(estimateW * joint_proxy)
    """
    agg_dir = os.path.join(base_dir, dataset_name, "results", "aggregated_results")
    if not os.path.exists(agg_dir):
        print(f"[Warn] aggregated_results dir not found: {agg_dir}")
        return pd.DataFrame()

    records = []
    
    # 建立映射缓存
    files = [f for f in os.listdir(agg_dir) if f.endswith(".csv")]
    file_map = {}
    for fname in files:
        base = fname.replace("aggregated_list_", "").replace(".csv", "")
        q_name = f"{base}.graph"
        file_map[q_name] = fname

    print(f"[Analysis] Extracting features for {len(queries)} queries...")
    
    for q in queries:
        q_name = q if q.endswith(".graph") else f"{q}.graph"
        fname = file_map.get(q_name)
        
        if not fname:
            continue

        fpath = os.path.join(agg_dir, fname)
        try:
            # 读取所有列，因为我们需要计算 proxy
            df_csv = pd.read_csv(fpath)
            
            # 确定 estimateW 列
            w_col = "estimateW" if "estimateW" in df_csv.columns else "a"
            
            # 确定 Proxy 列 (根据您的实际 CSV 列名修改)
            # 这里假设 CSV 中包含这些列，或者您需要根据实际情况调整列名
            # 常见的列名可能是: ML1_proxy4b1_probability, ML2_proxy1_probability
            post_cols = [c for c in df_csv.columns if "ML1_" in c and "proxy" in c]
            comment_cols = [c for c in df_csv.columns if "ML2_" in c and "proxy" in c]
            
            post_proxy_col = post_cols[0] if post_cols else "ML1_proxy4b1_probability"
            comment_proxy_col = comment_cols[0] if comment_cols else "ML2_proxy1_probability"

            # 1. Core Instances (W > 0)
            df_csv[w_col] = pd.to_numeric(df_csv[w_col], errors="coerce").fillna(0.0)
            df_valid = df_csv[df_csv[w_col] > 0].copy()
            core_cnt = len(df_valid)
            
            if core_cnt > 0:
                # 2. Global Estimate W
                global_w = df_valid[w_col].sum()
                
                # 3. Joint Proxy
                # 需要逐行计算联合概率
                df_valid['joint_proxy'] = df_valid.apply(
                    lambda row: calculate_joint_proxy(row, post_proxy_col, comment_proxy_col), axis=1
                )
                
                avg_proxy = df_valid['joint_proxy'].mean()
                
                # 4. Total Importance Weight
                df_valid['importance'] = df_valid[w_col] * df_valid['joint_proxy']
                total_imp = df_valid['importance'].sum()
                
            else:
                global_w = 0.0
                avg_proxy = 0.0
                total_imp = 0.0

            records.append({
                "query_basename": q_name,
                "core_instances": core_cnt,
                "global_estimateW": global_w,
                "avg_joint_proxy": avg_proxy,
                "total_importance_weight": total_imp
            })
            
        except Exception as e:
            print(f"Error reading {fname}: {e}")

    return pd.DataFrame(records)

def visualize_analysis_enhanced(df_merged, dataset_name, budget_frac, out_dir):
    """
    增强版可视化：绘制 Error Diff 与多个特征的关系
    """
    if df_merged.empty:
        return

    # 添加分组标签
    df_merged["Group"] = df_merged.apply(
        lambda x: "NRS Better" if x["nrs_minus_rs"] < 0 else "RS Better/Equal", axis=1
    )
    
    # 定义要分析的特征列表
    features = [
        ("core_instances", "Core Instances Count"),
        ("global_estimateW", "Global Estimate W (Sum of W)"),
        ("avg_joint_proxy", "Average Joint Proxy Score"),
        ("total_importance_weight", "Total Importance Weight (Sum of W*P)")
    ]

    for feat_col, feat_name in features:
        if feat_col not in df_merged.columns:
            continue
            
        plt.figure(figsize=(10, 6))
        
        # 使用对数坐标处理数值跨度大的特征
        if feat_col in ["core_instances", "global_estimateW", "total_importance_weight"]:
             x_data = np.log10(df_merged[feat_col] + 1)
             x_label = f"log10({feat_name})"
        else:
             x_data = df_merged[feat_col]
             x_label = feat_name

        sns.scatterplot(
            x=x_data, 
            y=df_merged["nrs_minus_rs"], 
            hue=df_merged["Group"], 
            palette={"NRS Better": "green", "RS Better/Equal": "red"},
            alpha=0.7,
            s=80
        )
        
        plt.axhline(0, color='black', linestyle='--', linewidth=1)
        plt.title(f"[{dataset_name} | Budget {budget_frac}] Error Diff vs {feat_name}")
        plt.xlabel(x_label)
        plt.ylabel("Diff (FOIS_nrs - FOIS_rs)\n< 0 means NRS is better")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        fname = f"scatter_diff_vs_{feat_col}_{budget_frac}.png"
        plt.savefig(os.path.join(out_dir, fname))
        plt.show()
        plt.close()
        
    print(f"[Vis] Enhanced plots saved to {out_dir}")

def analyze_nrs_vs_rs_performance(dataset_name: str,
                                  budget_frac: float,
                                  base_dir="/home/wangshuo/resource/datasets/parler_data"):
    
    # 1. 确定输出目录
    out_dir = os.path.join(base_dir, dataset_name, "results", "analysis_nrs_vs_rs_features")
    os.makedirs(out_dir, exist_ok=True)

    # 2. 读取 Budget Curve CSV
    # 优先查找 efficienty 目录，其次 budget_curve 目录
    paths_to_check = [
        os.path.join(base_dir, dataset_name, "results", "efficiency", "FOIS_rs_POSS_budget_curve.csv"),
        os.path.join(base_dir, dataset_name, "results", "budget_curve", "FOIS_rs_POSS_budget_curve.csv")
    ]
    curve_path = None
    for p in paths_to_check:
        if os.path.exists(p):
            curve_path = p
            break
    
    if not curve_path:
        print(f"[Error] Budget curve CSV not found.")
        return

    print(f"[Load] Reading {curve_path}...")
    df = pd.read_csv(curve_path)
    if "query_name" in df.columns: df.rename(columns={"query_name": "query_basename"}, inplace=True)
    
    # 过滤采样率
    df["budget_frac"] = pd.to_numeric(df["budget_frac"], errors="coerce")
    df_sub = df[np.isclose(df["budget_frac"], budget_frac, atol=1e-9)].copy()
    
    # 3. 过滤白名单
    whitelist = _load_gt_whitelist(dataset_name, base_dir=base_dir)
    if whitelist:
        df_sub["query_basename"] = df_sub["query_basename"].astype(str).apply(
            lambda x: x if x.endswith(".graph") else f"{x}.graph"
        )
        df_sub = df_sub[df_sub["query_basename"].isin(whitelist)]

    # 4. 计算平均误差
    if "Qerror" not in df_sub.columns: df_sub.rename(columns={"abs_error": "Qerror"}, inplace=True)

    target_methods = ["FOIS_nrs", "FOIS_rs"]
    df_agg = (
        df_sub[df_sub["method"].isin(target_methods)]
        .groupby(["query_basename", "method"])["Qerror"]
        .mean()
        .reset_index()
    )

    df_pivot = df_agg.pivot(index="query_basename", columns="method", values="Qerror").reset_index()
    df_pivot.dropna(subset=target_methods, inplace=True)
    df_pivot["nrs_minus_rs"] = df_pivot["FOIS_nrs"] - df_pivot["FOIS_rs"]

    # 5. 【核心修改】提取高级特征
    queries_to_analyze = df_pivot["query_basename"].tolist()
    df_features = extract_features_for_queries(dataset_name, queries_to_analyze, base_dir)
    
    # 合并
    if not df_features.empty:
        df_final = df_pivot.merge(df_features, on="query_basename", how="left")
    else:
        print("[Error] Failed to extract features.")
        return

    # 6. 保存与可视化
    cols = ["query_basename", "FOIS_nrs", "FOIS_rs", "nrs_minus_rs", 
            "core_instances", "global_estimateW", "avg_joint_proxy", "total_importance_weight"]
    
    # 排序并打印
    df_final.sort_values("nrs_minus_rs", inplace=True)
    print("\nTop 5 Queries where NRS is Better (Negative Diff):")
    print(df_final[cols].head(5).to_string(index=False))
    
    print("\nTop 5 Queries where RS is Better (Positive Diff):")
    print(df_final[cols].tail(5).to_string(index=False))

    # 保存
    csv_out = os.path.join(out_dir, f"nrs_vs_rs_features_budget_{budget_frac}.csv")
    df_final.to_csv(csv_out, index=False)
    print(f"\n[Save] Detailed analysis saved to {csv_out}")

    # 绘图
    visualize_analysis_enhanced(df_final, dataset_name, budget_frac, out_dir)


if __name__ == "__main__":
    DATASET_NAME = "dataset_three"   
    BUDGET_FRAC = 0.10              
    
    analyze_nrs_vs_rs_performance(
        dataset_name=DATASET_NAME,
        budget_frac=BUDGET_FRAC
    )

### 3. 
读取数据集dataset_test,采样率为0.1 下, 对于每个查询,allocation_strategy_comparison.csv 中各个方法的平均误差>[0.10,0.15,0.20]的查询有哪些,并输出查询对应的核心集合的大小,每个查询核心集大小保存到了D_count.csv文件中, 文件部分内容如下, 
dataset_name,query_basename,total_rows,core_instances
dataset_test,query_cycle_4_0.graph,5133,5133
dataset_test,query_cycle_4_1.graph,9108,9108
dataset_test,query_cycle_4_10.graph,3904,3904
dataset_test,query_cycle_4_11.graph,6536,6536
dataset_test,query_cycle_4_12.graph,5021,5021
dataset_test,query_cycle_4_13.graph,9090,9090

In [4]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 1. 配置参数
# ==========================================
dataset_name = 'dataset_test'
base_dir = f"/home/wangshuo/resource/datasets/parler_data/{dataset_name}"
results_dir = os.path.join(base_dir, "results", "efficiency")

# 输入文件路径
path_alloc = os.path.join(results_dir, "allocation_strategy_comparison3.csv")
# path_alloc = os.path.join(results_dir, "FOIS_rs_FOSS_nrs_budget_curve_fast.csv")
# 假设 D_count.csv 在 results 目录下，如果位置不同请修改此处
path_count = '/home/wangshuo/projects/Neo4j_Exp/pythonProject/src/paper_exp/D_count.csv'

# 分析配置
TARGET_BUDGET = 0.1
THRESHOLDS = [0.10, 0.15, 0.20]

# ==========================================
# 2. 数据读取与预处理
# ==========================================
print(f"--- 开始分析数据集: {dataset_name} (Budget={TARGET_BUDGET}) ---")

# 2.1 读取结果文件
if not os.path.exists(path_alloc):
    print(f"[Error] 结果文件未找到: {path_alloc}")
    exit()
df_res = pd.read_csv(path_alloc)

# 2.2 读取核心集计数文件
if not os.path.exists(path_count):
    print(f"[Error] 核心集计数文件未找到: {path_count}")
    # 如果没有计数文件，我们可以创建一个假的以便代码不报错，只是大小显示为 N/A
    print("将继续分析误差，但无法显示核心集大小。")
    df_count = pd.DataFrame(columns=["query_basename", "core_instances"])
else:
    df_count = pd.read_csv(path_count)

# ==========================================
# 3. 数据清洗与对齐
# ==========================================

# 3.1 统一 Query 名称 (移除 .graph 后缀)
def clean_query_name(name):
    return str(name).replace(".graph", "").strip()

df_res["query_basename"] = df_res["query_basename"].apply(clean_query_name)
if not df_count.empty:
    df_count["query_basename"] = df_count["query_basename"].apply(clean_query_name)

# 3.2 筛选目标采样率 (0.1)
# 使用 np.isclose 处理浮点数精度问题
mask_budget = df_res["budget_frac"].apply(lambda x: np.isclose(x, TARGET_BUDGET, atol=1e-4))
df_target = df_res[mask_budget].copy()

if df_target.empty:
    print(f"[Error] 在 CSV 中未找到 budget_frac = {TARGET_BUDGET} 的数据。")
    exit()

# 3.3 计算 ARE (绝对相对误差)
# 也就是 Qerror，如果 CSV 里有 Qerror 直接用，没有则重算
if "Qerror" in df_target.columns:
    df_target["ARE"] = df_target["Qerror"]
else:
    df_target = df_target[df_target["T_true"] != 0]
    df_target["ARE"] = ((df_target["T_hat"] - df_target["T_true"]) / df_target["T_true"]).abs()

# 3.4 聚合多次运行结果 (取平均误差)
# 按照 方法 和 查询名 分组
df_agg = df_target.groupby(["method", "query_basename"])["ARE"].mean().reset_index()
df_agg.rename(columns={"ARE": "Mean_ARE"}, inplace=True)

# 3.5 合并核心集大小信息
if not df_count.empty:
    # 只需要 query_basename 和 core_instances (或者 total_rows，看你具体指哪个)
    # 根据你的描述，D_count header: dataset_name,query_basename,total_rows,core_instances
    # 通常 core_instances 和 total_rows 在这种语境下是一样的 (CSV行数)
    cols_to_merge = ["query_basename", "core_instances"] 
    df_merged = pd.merge(df_agg, df_count[cols_to_merge], on="query_basename", how="left")
else:
    df_merged = df_agg
    df_merged["core_instances"] = -1

# 填充未匹配到的核心集大小 (防止报错)
df_merged["core_instances"] = df_merged["core_instances"].fillna(-1).astype(int)

# ==========================================
# 4. 阈值筛选与输出
# ==========================================

methods = sorted(df_merged["method"].unique())

print("\n" + "="*80)
print(f"分析报告: 采样率 {TARGET_BUDGET} 下的高误差查询")
print(f"核心集大小来源: {path_count}")
print("="*80)

for method in methods:
    print(f"\n>>> 方法: {method}")
    subset = df_merged[df_merged["method"] == method]
    
    for threshold in THRESHOLDS:
        # 筛选大于阈值的查询
        bad_queries = subset[subset["Mean_ARE"] > threshold].sort_values(by="Mean_ARE", ascending=False)
        
        count = len(bad_queries)
        print(f"\n  [阈值 > {threshold:.2f}] 共发现 {count} 个查询:")
        
        if count > 0:
            # 打印表头
            print(f"    {'Query Name':<35} | {'Mean Error':<12} | {'Core Size (N)':<15}")
            print(f"    {'-'*35} | {'-'*12} | {'-'*15}")
            
            for _, row in bad_queries.iterrows():
                q_name = row['query_basename']
                err = row['Mean_ARE']
                size = row['core_instances']
                
                size_str = f"{size}" if size != -1 else "N/A"
                print(f"    {q_name:<35} | {err:.4f}       | {size_str:<15}")
        else:
            print("    (无)")

    print("-" * 60)

# ==========================================
# 5. 可选：保存到文件
# ==========================================
# output_path = os.path.join(results_dir, "high_error_analysis.csv")
# df_merged.to_csv(output_path, index=False)
# print(f"\n完整分析数据已保存至: {output_path}")

--- 开始分析数据集: dataset_test (Budget=0.1) ---

分析报告: 采样率 0.1 下的高误差查询
核心集大小来源: /home/wangshuo/projects/Neo4j_Exp/pythonProject/src/paper_exp/D_count.csv

>>> 方法: 3_ProxyE_Imp_RootWP

  [阈值 > 0.10] 共发现 52 个查询:
    Query Name                          | Mean Error   | Core Size (N)  
    ----------------------------------- | ------------ | ---------------
    query_cycle_6_17                    | 1.4331       | 1429           
    query_cycle_8_12                    | 1.0500       | 664            
    query_cycle_8_41                    | 1.0000       | 4088           
    query_cycle_8_19                    | 0.3758       | 5879           
    query_cycle_8_0                     | 0.3384       | 2201           
    query_cycle_8_26                    | 0.3116       | 3170           
    query_cycle_8_46                    | 0.3019       | 8623           
    query_cycle_6_27                    | 0.2565       | 1045           
    query_cycle_8_23                    | 0.2367       | 1579    